# 10 · LLM Mathematics

**The big idea:** an LLM is a function from "text so far" to a probability distribution over the next token. Everything else (training loss, perplexity, sampling) is math on that distribution.

---

## 📂 Notes in this chapter

One short note per concept, grouped into folders. Read in order. Code builds on earlier notes, so `notebook.ipynb` (all code, in order) is the easiest way to run everything.

### `1-probability-and-loss/`

- [1. Language-Model Probability](1-probability-and-loss/01-language-model-probability.md)
- [2. Next-Token Prediction](1-probability-and-loss/02-next-token-prediction.md)
- [3. Cross-Entropy Objective & Log-Likelihood](1-probability-and-loss/03-cross-entropy-objective-log-likelihood.md)
- [4. Perplexity](1-probability-and-loss/04-perplexity.md)

### `2-sampling/`

- [5. Sampling & Temperature](2-sampling/05-sampling-temperature.md)
- [6. Top-k and Top-p (Nucleus)](2-sampling/06-top-k-and-top-p-nucleus.md)

### `3-similarity-context-inference/`

- [7. Embedding Similarity](3-similarity-context-inference/07-embedding-similarity.md)
- [8. Context Windows](3-similarity-context-inference/08-context-windows.md)
- [9. Basic LLM Inference Mathematics](3-similarity-context-inference/09-basic-llm-inference-mathematics.md)

---
**Full loop of the course:** vectors → matrices → gradients → probability → cross-entropy → softmax → attention → next-token prediction. 🎉


# Language-Model Probability

*📓 Notes · [10 · LLM Mathematics](../README.md) · probability and loss*

**📝 In simple words:** The probability of a sentence, built with the chain rule of probability (03).

**📐 Formula to remember:** `P(x₁…x_T) = Π P(xₜ | x₁…xₜ₋₁)`

**🔢 Small example:** `"I love AI"`: `P(I)=0.1`, `P(love | I)=0.2`, `P(AI | I love)=0.5`.

**✏️ Let me calculate it:** `0.1 × 0.2 × 0.5 = 0.01`. Take logs so it's a sum: `ln 0.1 + ln 0.2 + ln 0.5 = −4.605`.

**🤖 Why AI uses it:** this factorization is *why* next-token prediction is enough to model all text.

In [ ]:
import numpy as np
probs = [0.1, 0.2, 0.5]
print(np.prod(probs), np.sum(np.log(probs)))   # 0.01  -4.605

**🧪 Try it myself:** Probabilities `[0.5, 0.5, 0.5, 0.5]`: sentence probability? <details><summary>Solution</summary>`1/16 = 0.0625`.</details>

---
[Next-Token Prediction](../1-probability-and-loss/02-next-token-prediction.md) →


# Next-Token Prediction

*📓 Notes · [10 · LLM Mathematics](../README.md) · probability and loss*

**📝 In simple words:** The model outputs one score (logit) per vocab token; softmax makes probabilities.

**🔢 Small example:** vocab `[cat, dog, car]`, logits `[2, 1, 0]`.

**✏️ Let me calculate it:** softmax → `[0.665, 0.245, 0.090]`. The model thinks "cat" is most likely.

In [ ]:
def softmax(z):
    e = np.exp(z - np.max(z)); return e / e.sum()
p = softmax(np.array([2., 1., 0.])); print(p.round(3))

**🧪 Try it myself:** Logits `[0,0,0]`: probabilities? <details><summary>Solution</summary>`[⅓, ⅓, ⅓]`.</details>

---
← [Language-Model Probability](../1-probability-and-loss/01-language-model-probability.md)  |  [Cross-Entropy Objective & Log-Likelihood](../1-probability-and-loss/03-cross-entropy-objective-log-likelihood.md) →


# Cross-Entropy Objective & Log-Likelihood

*📓 Notes · [10 · LLM Mathematics](../README.md) · probability and loss*

**📝 In simple words:** Training pushes up the probability of the *actual* next token.

**📐 Formula to remember:** `L = −(1/T) Σₜ log P(xₜ | x<ₜ)`: the negative average log-likelihood (06, 07).

**✏️ Let me calculate it:** truth is "dog" → `−ln 0.245 = 1.406`. After training makes it `0.8` → `0.223`.

**🤖 Why AI uses it:** this single loss, averaged over trillions of tokens, is how GPT-style models are trained. Gradient w.r.t. logits is `p − onehot` (08).

In [ ]:
print(-np.log(p[1]), -np.log(0.8))

**🧪 Try it myself:** Loss if the model gave the true token probability 1? <details><summary>Solution</summary>`−ln 1 = 0`.</details>

---
← [Next-Token Prediction](../1-probability-and-loss/02-next-token-prediction.md)  |  [Perplexity](../1-probability-and-loss/04-perplexity.md) →


# Perplexity

*📓 Notes · [10 · LLM Mathematics](../README.md) · probability and loss*

**📝 In simple words:** `e^(average loss)`: the effective number of choices the model hesitates between.

**📐 Formula to remember:** `PPL = exp(−(1/T) Σ log P(xₜ | x<ₜ))`

**✏️ Let me calculate it:** from step 1: average `= 4.605/3 = 1.535` → `PPL = e^1.535 = 4.64`. A model that is uniform over 100 tokens has `PPL = 100`.

**🤖 Why AI uses it:** the standard intrinsic metric; lower is better.

In [ ]:
nll = -np.mean(np.log(probs)); print(np.exp(nll))   # 4.64
print(np.exp(-np.log(1/100)))                        # 100

**🧪 Try it myself:** Average loss 2.0 nats: PPL? <details><summary>Solution</summary>`e² ≈ 7.39`.</details>

---
← [Cross-Entropy Objective & Log-Likelihood](../1-probability-and-loss/03-cross-entropy-objective-log-likelihood.md)  |  [Sampling & Temperature](../2-sampling/05-sampling-temperature.md) →


# Sampling & Temperature

*📓 Notes · [10 · LLM Mathematics](../README.md) · sampling*

**📝 In simple words:** Pick the next token from `p`; temperature reshapes `p` first.

**📐 Formula to remember:** `pᵢ = softmax(zᵢ / T)`

**✏️ Let me calculate it (`z = [2,1,0]`):**

```text
T = 0.5 → z/T=[4,2,0] → [0.867, 0.117, 0.016]   sharper, safer
T = 1   →            → [0.665, 0.245, 0.090]
T = 2   → [1,0.5,0]  → [0.506, 0.307, 0.186]   flatter, more creative
T → 0   → always the top token (greedy)
```

**🤖 Why AI uses it:** controls the creativity/reliability trade-off (it changes the *entropy* of the output, 06).

In [ ]:
z = np.array([2., 1., 0.])
for T in (0.5, 1, 2): print(T, softmax(z / T).round(3))
rng = np.random.default_rng(0)
print(rng.choice(["cat", "dog", "car"], p=softmax(z)))

**🧪 Try it myself:** What does T = 1 do to the distribution? <details><summary>Solution</summary>Nothing. It's the plain softmax.</details>

---
← [Perplexity](../1-probability-and-loss/04-perplexity.md)  |  [Top-k and Top-p (Nucleus)](../2-sampling/06-top-k-and-top-p-nucleus.md) →


# Top-k and Top-p (Nucleus)

*📓 Notes · [10 · LLM Mathematics](../README.md) · sampling*

**📝 In simple words:** Cut off the unlikely tail, renormalize, then sample.
- **Top-k:** keep the `k` most probable tokens.
- **Top-p:** keep the smallest set whose cumulative probability ≥ `p`.

**✏️ Let me calculate it (`p = [0.665, 0.245, 0.090]`):** top-k `k=2` → keep `0.665, 0.245` → divide by `0.910` → `[0.731, 0.269]`. Top-p `p=0.8`: cumulative `0.665 → 0.910 ≥ 0.8` → same two tokens.

**🤖 Why AI uses it:** stops rare, nonsensical tokens from being sampled while keeping variety.

In [ ]:
def top_k(p, k):
    q = np.zeros_like(p); idx = np.argsort(p)[-k:]; q[idx] = p[idx]; return q / q.sum()
def top_p(p, thr):
    order = np.argsort(p)[::-1]; cum = np.cumsum(p[order])
    keep = order[: np.searchsorted(cum, thr) + 1]
    q = np.zeros_like(p); q[keep] = p[keep]; return q / q.sum()
print(top_k(p, 2).round(3), top_p(p, 0.8).round(3))

**🧪 Try it myself:** Top-p with `p=0.6` on the distribution above? <details><summary>Solution</summary>Only the first token is needed (0.665 ≥ 0.6) → `[1, 0, 0]`.</details>

---
← [Sampling & Temperature](../2-sampling/05-sampling-temperature.md)  |  [Embedding Similarity](../3-similarity-context-inference/07-embedding-similarity.md) →


# Embedding Similarity

*📓 Notes · [10 · LLM Mathematics](../README.md) · similarity context inference*

**📝 In simple words:** Meaning compared by cosine similarity between vectors (01).

**📐 Formula to remember:** `cos(a,b) = a·b / (‖a‖‖b‖)`

**🔢 Small example:** `king=[0.9,0.1]`, `queen=[0.8,0.3]`, `apple=[0.1,0.9]`.

**✏️ Let me calculate it:** `cos(king,queen) = 0.75/(0.906·0.854) = 0.97`; `cos(king,apple) = 0.18/(0.906·0.906) = 0.22`.

**🤖 Why AI uses it:** semantic search and retrieval (RAG) find the nearest embeddings to a query.

In [ ]:
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
king, queen, apple = np.array([.9, .1]), np.array([.8, .3]), np.array([.1, .9])
print(cos(king, queen).round(2), cos(king, apple).round(2))   # 0.97 0.22

**🧪 Try it myself:** Cosine similarity of a vector with itself? <details><summary>Solution</summary>1.</details>

---
← [Top-k and Top-p (Nucleus)](../2-sampling/06-top-k-and-top-p-nucleus.md)  |  [Context Windows](../3-similarity-context-inference/08-context-windows.md) →


# Context Windows

*📓 Notes · [10 · LLM Mathematics](../README.md) · similarity context inference*

**📝 In simple words:** The maximum number of tokens the model can attend to at once.

**Why it costs so much:** attention builds an `n × n` score matrix, so cost grows with `n²`.

**✏️ Let me calculate it:** `n = 1,000` → 1,000,000 scores per head per layer; `n = 2,000` → 4,000,000 (**4×** for 2× the text).

**🤖 Why AI uses it:** it's the practical limit on how much text the model can "see"; long-context work is largely about making this cheaper.

In [ ]:
for n in (1000, 2000, 4000): print(n, n * n)

**🧪 Try it myself:** Tripling the context multiplies attention cost by? <details><summary>Solution</summary>9.</details>

---
← [Embedding Similarity](../3-similarity-context-inference/07-embedding-similarity.md)  |  [Basic LLM Inference Mathematics](../3-similarity-context-inference/09-basic-llm-inference-mathematics.md) →


# Basic LLM Inference Mathematics

*📓 Notes · [10 · LLM Mathematics](../README.md) · similarity context inference*

**Generation loop:** (1) run the model on the prompt → logits (2) pick a token (sampling) (3) append it (4) repeat.

**KV cache:** keys/values of earlier tokens don't change, so we store them instead of recomputing.

**Memory for the KV cache:** `2 × layers × tokens × d_model × bytes`.

**✏️ Let me calculate it:** 32 layers, `d_model = 4096`, 1,000 tokens, fp16 (2 bytes): `2·32·1000·4096·2 = 524,288,000 B ≈ 0.52 GB`.

**Compute per token:** roughly `2 × N` floating-point operations for `N` parameters: a 7B model ≈ 14 GFLOPs per generated token.

**Weights memory:** `N × bytes`: 7B in fp16 = 14 GB; in 4-bit ≈ 3.5 GB.

In [ ]:
layers, d, n, bytes_ = 32, 4096, 1000, 2
print(2 * layers * n * d * bytes_ / 1e9, "GB KV cache")
print(7e9 * 2 / 1e9, "GB of fp16 weights")

**🧪 Try it myself:** KV cache for the same model with 4,000 tokens? <details><summary>Solution</summary>4× larger ≈ 2.1 GB.</details>

---
← [Context Windows](../3-similarity-context-inference/08-context-windows.md)